# v9d: learning a NEW task at a LOWER learning rate, 30% switched off (GPU 1)
Same as `newtask_v9.ipynb`, with three changes:
1. learning rate **3e-5** and **1e-4** (fine-tuning rates; last run used 2e-4, which wiped out translation)
2. **30%** of each layer's working neurons switched off (was 10%)
3. start models: distant **phase 20** (not 60, which had lost its ability to learn) and the related run's last phase.

Tasks: Indonesian (new language) and addition (not language). Conditions: nothing / self / look-alike / random; fresh model as a control.
Results: `analysis/v9/newtask_lr3e-5_d30/` and `analysis/v9/newtask_lr1e-4_d30/`. Seed 0 of everything first.
**Check first:** the FORGETTING table should now show small rises (well below the 5-9 of last time).

In [ ]:
import os, sys, glob, json, time, subprocess
os.environ["HF_HUB_DISABLE_XET"] = "1"
GPU = "1"          # second GPU (newtask_v9.ipynb uses GPU 0)
REPO = "/workspace/shreshth"; os.chdir(REPO); sys.path.insert(0, REPO)
from IPython.display import Image, display
from huggingface_hub import HfApi, hf_hub_download
for attempt in range(5):
    try:
        api = HfApi(); HF_REPO = api.whoami()["name"] + "/cml-lang-self-checkpoints"; FILES = api.list_repo_files(HF_REPO); break
    except Exception as e:
        print("HF retry", attempt, type(e).__name__); time.sleep(10)

def get(run, phases):
    for ph in phases:
        if glob.glob(f"runs/{run}/ckpt/p{ph:03d}_*.pt"):
            continue
        name = [f for f in FILES if f.startswith(f"{run}/ckpt/p{ph:03d}_")]
        if name:
            hf_hub_download(HF_REPO, name[0], local_dir="runs")
        else:
            print("missing (not here, not on HF):", run, ph)

def last_phase(run):
    phs = [int(os.path.basename(f)[1:4]) for f in FILES if f.startswith(f"{run}/ckpt/p")]
    phs += [int(os.path.basename(f)[1:4]) for f in glob.glob(f"runs/{run}/ckpt/p*.pt")]
    return max(phs)

def run(langset, out, lr, *cmd):
    env = {**os.environ, "CML_LANGSET": langset}
    p = subprocess.Popen([sys.executable, "-u", "newtask_v9.py", "--gpu", GPU, "--tasks", "indonesian,addition", "--out", out,
                          "--lr", lr, "--dose", "0.30", *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout:
        print(line, end="")
    p.wait(); print("exit code", p.returncode)

DIST, REL = "v4b_continual_s0", "v8_indic_continual_s0"
PD, PR = 20, last_phase(REL)
get(DIST, [PD]); get(REL, [PR])
RUNS = [("3e-5", "newtask_lr3e-5_d30"), ("1e-4", "newtask_lr1e-4_d30")]
print(f"start models: distant = {DIST} phase {PD} | related = {REL} phase {PR}")

In [ ]:
need = [f"data/en-{l}/train.jsonl" for l in ["hi", "zh", "gu"]] + [f"data/flores/devtest.{l}.jsonl" for l in ["en", "hi", "zh", "gu"]]
need += [f"data/flores/dev.{l}.jsonl" for l in ["en", "hi", "zh", "ar", "gu", "pa"]] + [f"data/en-{l}/val.jsonl" for l in ["hi", "zh", "ar", "gu", "pa"]]
miss = [f for f in need if not os.path.exists(f)]
print("all there" if not miss else f"MISSING: {miss}  -> run your download snippet first, then re-run this cell")

In [ ]:
subprocess.run([sys.executable, "-u", "prep_newtasks.py"], check=True)

In [ ]:
for s in ["0", "1", "2"]:
    for lr, out in RUNS:
        run("distant", out, lr, "--start", f"distant={DIST}:{PD}", "--seeds", s)
        run("distant", out, lr, "--start", "fresh", "--seeds", s)
        run("indic", out, lr, "--start", f"related={REL}:{PR}", "--seeds", s)
        subprocess.run([sys.executable, "newtask_v9.py", "--summarize", "--out", out])

In [ ]:
subprocess.run([sys.executable, "newtask_v9.py", "--summarize", "--out", "newtask_lr3e-5_d30"])
for f in ["fig1_learning_curves_loss.png", "fig2_learning_curves_score.png", "fig3_near_to_far.png"]:
    display(Image(f"analysis/v9/newtask_lr3e-5_d30/{f}"))

In [ ]:
subprocess.run([sys.executable, "newtask_v9.py", "--summarize", "--out", "newtask_lr1e-4_d30"])
for f in ["fig1_learning_curves_loss.png", "fig2_learning_curves_score.png", "fig3_near_to_far.png"]:
    display(Image(f"analysis/v9/newtask_lr1e-4_d30/{f}"))